[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [1]:
import torch

In [17]:
# ✏️ YOUR IMPLEMENTATION HERE

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    beams = [[start_token]]
    cum_probs = [0.0]
    for i in range(max_len):
        candidates = []
        for b in range(len(beams)):
            logits = log_prob_fn(beams[b])
            vals, idx = logits.topk(beam_width,dim=-1)
            candidates.extend(
                [
                    (cum_probs[b]+vals[j].item(),beams[b]+[idx[j]]) 
                    for j in range(beam_width)
                ]
            )
        candidates.sort(reverse=True)
        cum_probs,beams = zip(*candidates[:beam_width])
        if beams[0][-1] == eos_token:
            return beams[0]
    return beams[0]

In [18]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, tensor(1), tensor(2), tensor(3), tensor(4)]


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
[(tensor(0.), [0, tensor(8)]), (tensor(0.), [0, tensor(7)]), (tensor(0.), [0, tensor(6)])]
([0, tensor(8)], [0, tensor(7)], [0, tensor(6)]) (tensor(0.), tensor(0.), tensor(0.))
[(tensor(0.), [0, tensor(8), tensor(8)]), (tensor(0.), [0, tensor(8), tensor(7)]), (tensor(0.), [0, tensor(8), tensor(6)]), (tensor(0.), [0, tensor(7), tensor(8)]), (tensor(0.), [0, tensor(7), tensor(7)]), (tensor(0.), [0, tensor(7), tensor(6)]), (tensor(0.), [0, tensor(6), tensor(8)]), (tensor(0.), [0, tensor(6), tensor(7)]), (tensor(0.), [0, tensor(6), tensor(6)])]
([0, tensor(8), tensor(8)], [0, tensor(8), tensor(7)], [0, tensor(8), tensor(6)]) (tensor(0.), tensor(0.), tensor(0.))
[(tensor(0.), [0, tensor(8), tensor(8), tensor(8)]), (tensor(0.), [0, tensor(8), tensor(8), tensor(7)]), (tensor(0.), [0, tensor(8), tensor(8), tensor(6)]), (tensor(0.), [0, tensor(8), tensor(7), tensor(8)]), (tensor(0.), [0, tensor(8), tens